# SPLADE EP1 — V5: generalização

Encoder SPLADE-PT-BR congelado + regressão logística do scikit-learn. A busca compara regularização e normalização L2. A representação, os pesos, o tokenizer e o comprimento máximo permanecem iguais à V4; o cache CSR `float32` da V4 é compatível.

1. Execute a instalação e reinicie a sessão se houve alteração de dependências.
2. Disponibilize `dados/train.xlsx` e `dados/test1.xlsx`.
3. Se tiver o cache anterior, copie a pasta `cache_splade/` para este ambiente antes da célula de representação.
4. Execute as demais células em ordem. A versão ativa deve ser `SPLADE_EP1_COLAB_V5_GENERALIZACAO`.

Validação: **10 folds externos**, com **3 folds internos** para escolher os parâmetros somente dentro do treino de cada fold externo. O controle `C=1`, sem normalização, usa as mesmas divisões externas.

São 9 configurações: `C=[0.001, 0.01, 0.1, 1]` sem normalização e `C=[0.01, 0.1, 1, 10, 100]` com L2. `class_weight="balanced"`, penalização L2, solver SAGA e tolerância são mantidos. A normalização altera a escala, por isso as faixas de C diferem. A execução completa envolve **318 ajustes** (incluindo a busca final), antes de eventuais novas tentativas de convergência. Os ajustes são sequenciais para limitar RAM; o classificador executa na CPU.

Cada fold concluído é salvo e pode ser retomado. Para preservar cache/checkpoints após excluir o ambiente, salve `cache_splade/` e `resultados_splade_v5/` no Drive e restaure as pastas na próxima sessão. Os checkpoints são identificados pelos dados e pela configuração.

Esta V5 foi validada localmente com uma execução reduzida de integração; as métricas completas devem ser obtidas no seu Colab. Não há resultados de acurácia pré-preenchidos neste notebook.


## 1. Instalação das versões verificadas

Esta célula executa a instalação; não está comentada. Depois dela, reinicie o kernel antes de prosseguir. As versões serão conferidas antes de carregar qualquer modelo.

A versão base de PyTorch é 2.11.0. Uma instalação existente `2.11.0+cpu` ou `2.11.0+cu...` atende ao requisito sem trocar o backend. O teste local usa CPU. A versão de Python do teste é 3.12.14; o notebook enviado executava em Python 3.13.15.


In [1]:
import subprocess
import sys

REQUIRED_VERSIONS_INSTALL = {'torch': '2.11.0', 'transformers': '4.57.3', 'accelerate': '1.15.0', 'huggingface_hub': '0.36.2', 'tokenizers': '0.22.2', 'safetensors': '0.8.0', 'numpy': '2.3.5', 'pandas': '2.2.3', 'scipy': '1.17.0', 'scikit-learn': '1.8.0', 'tqdm': '4.70.1', 'psutil': '7.2.2', 'openpyxl': '3.1.5', 'setuptools': '81.0.0'}
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--upgrade",
    *[f"{package}=={version}" for package, version in REQUIRED_VERSIONS_INSTALL.items()],
])
print("Instalação concluída. REINICIE O KERNEL e execute a próxima seção desde o início.")


Instalação concluída. REINICIE O KERNEL e execute a próxima seção desde o início.


## Código e ambiente

O código está incorporado; nenhum `.py` externo é necessário. As dependências fixadas são as mesmas da V4 que concluiu a execução no seu Colab.


In [2]:
"""
EP1 - ACH2118 Introdução ao Processamento de Língua Natural
Abordagem: SPLADE-PT-BR como representação lexical esparsa aprendida
           + classificadores lineares do scikit-learn.

Este arquivo NÃO faz fine-tuning do SPLADE/BERTimbau.
O SPLADE-PT-BR é usado apenas como codificador congelado de texto.

Dependências fixadas em requirements_splade_ep1_v4.txt.
Instale as versões e reinicie o kernel antes de executar o notebook.
"""

from __future__ import annotations

import gc
import hashlib
import importlib.util
import json
import os
import platform
import sys
import time
from contextlib import nullcontext
from pathlib import Path
from typing import Callable

# Versões verificadas no ambiente de teste isolado do servidor.
CODE_VERSION = "SPLADE_EP1_COLAB_V5_GENERALIZACAO"
REQUIRED_VERSIONS = {'torch': '2.11.0', 'transformers': '4.57.3', 'accelerate': '1.15.0', 'huggingface_hub': '0.36.2', 'tokenizers': '0.22.2', 'safetensors': '0.8.0', 'numpy': '2.3.5', 'pandas': '2.2.3', 'scipy': '1.17.0', 'scikit-learn': '1.8.0', 'tqdm': '4.70.1', 'psutil': '7.2.2', 'openpyxl': '3.1.5', 'setuptools': '81.0.0'}

def validate_environment() -> dict[str, str]:
    from importlib.metadata import PackageNotFoundError, version
    module_names = {"scikit-learn": "sklearn", "huggingface_hub": "huggingface_hub"}
    found = {}
    errors = []
    for package, expected in REQUIRED_VERSIONS.items():
        try:
            actual = version(package)
        except PackageNotFoundError:
            actual = "ausente"
        found[package] = actual
        comparable = actual.split("+")[0] if package == "torch" else actual
        if comparable != expected:
            errors.append(f"{package}: instalado={actual}; esperado={expected}")
        module = sys.modules.get(module_names.get(package, package))
        loaded = getattr(module, "__version__", None) if module is not None else None
        if loaded is not None:
            loaded_comparable = loaded.split("+")[0] if package == "torch" else loaded
            if loaded_comparable != expected:
                errors.append(f"{package}: versão ainda carregada no kernel={loaded}")
    if errors:
        raise RuntimeError(
            "Ambiente diferente do verificado. Execute a célula 1 de instalação "
            "(ou pip install -r requirements_splade_ep1_v4.txt), reinicie o kernel "
            "e execute desde o início.\n" + "\n".join(errors)
        )
    return found

validate_environment()
# Esta solução usa exclusivamente o backend PyTorch.
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

import numpy as np
import pandas as pd
import scipy
from scipy import sparse
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm

import torch
import transformers
from huggingface_hub import hf_hub_download
from accelerate import init_empty_weights
from transformers import AutoModelForMaskedLM, AutoTokenizer, BertConfig, PreTrainedModel

try:
    import psutil
except ImportError:  # memória RSS vira opcional
    psutil = None


# -----------------------------------------------------------------------------
# Configuração
# -----------------------------------------------------------------------------
RANDOM_STATE = 42
N_SPLITS = 10
LABELS = ["c1", "c234", "c5"]

TRAIN_PATH = Path("dados/train.xlsx")
TEST_PATH = Path("dados/test1.xlsx")
CACHE_DIR = Path("cache_splade")
RESULTS_DIR = Path("resultados_splade_v5")
ACTIVE_RESULTS_DIR = RESULTS_DIR

MODEL_ID = "AxelPCG/splade-pt-br"
# Snapshot conhecido/reprodutível que contém modeling_splade.py e os pesos.
MODEL_REVISION = "fff5e6a3a3e0dc834227d4600a90a44025863bb6"
ENCODER_CACHE_VERSION = "v4_direct_checkpoint_torch211"
MAX_LENGTH = 256
BATCH_SIZE = 4
USE_CACHE = True

# Comparações recuperadas dos experimentos anteriores do projeto.
# Valores históricos de referência; nenhum foi reexecutado nesta versão.
PREVIOUS_RESULTS = [
    {
        "modelo": "Baseline TF-IDF + Regressão Logística",
        "acuracia_media": 0.4565,
        "folds": 10,
        "comparacao_direta": False,
        "observacao": "baseline anterior do projeto; treino médio ~69,52%",
    },
    {
        "modelo": "SetFit + CosineSimilarityLoss + LR balanced",
        "acuracia_media": 0.4012,
        "folds": 3,
        "comparacao_direta": False,
        "observacao": "protocolo de 3 folds; usar apenas como referência contextual",
    },
]


# -----------------------------------------------------------------------------
# Utilitários de logging/memória
# -----------------------------------------------------------------------------
def rss_gb() -> float | None:
    if psutil is None:
        return None
    return psutil.Process(os.getpid()).memory_info().rss / (1024**3)


def log(message: str) -> None:
    memoria = rss_gb()
    sufixo = f" | RSS≈{memoria:.2f} GB" if memoria is not None else ""
    print(f"[{time.strftime('%H:%M:%S')}] {message}{sufixo}", flush=True)


def sparse_memory_mb(matrix: sparse.spmatrix) -> float:
    csr = matrix.tocsr(copy=False)
    return (csr.data.nbytes + csr.indices.nbytes + csr.indptr.nbytes) / (1024**2)


def log_sparse_matrix(name: str, matrix: sparse.spmatrix) -> None:
    total = matrix.shape[0] * matrix.shape[1]
    sparsity = 1.0 - (matrix.nnz / total) if total else 1.0
    dense_mb = total * np.dtype(np.float32).itemsize / (1024**2)
    log(
        f"{name}: shape={matrix.shape}, nnz={matrix.nnz:,}, "
        f"esparsidade={sparsity:.4%}, CSR≈{sparse_memory_mb(matrix):.2f} MB, "
        f"denso float32≈{dense_mb:.2f} MB"
    )


def log_versions() -> None:
    print(f"Código ativo: {CODE_VERSION}")
    print(f"Python: {platform.python_version()}")
    for package, installed in validate_environment().items():
        print(f"  {package:<16}: {installed} [versão verificada]")
    print(f"Dispositivo: {'cuda' if torch.cuda.is_available() else 'cpu'}")


# -----------------------------------------------------------------------------
# Dados
# -----------------------------------------------------------------------------
def load_datasets(
    train_path: Path = TRAIN_PATH,
    test_path: Path = TEST_PATH,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str], np.ndarray, list[str]]:
    log(f"Carregando treino: {train_path}")
    train_df = pd.read_excel(train_path)
    log(f"Carregando teste: {test_path}")
    test_df = pd.read_excel(test_path)

    required_train = {"resp_text", "clarity"}
    if not required_train.issubset(train_df.columns):
        raise ValueError(f"Treino deve conter as colunas {sorted(required_train)}")
    if "resp_text" not in test_df.columns:
        raise ValueError("Teste deve conter a coluna 'resp_text'")

    train_df = train_df.copy()
    test_df = test_df.copy()
    train_df["resp_text"] = train_df["resp_text"].fillna("").astype(str)
    test_df["resp_text"] = test_df["resp_text"].fillna("").astype(str)
    train_df["clarity"] = train_df["clarity"].astype(str)

    y = train_df["clarity"].to_numpy()
    observed = set(np.unique(y))
    expected = set(LABELS)
    if observed != expected:
        raise ValueError(f"Rótulos encontrados={sorted(observed)}; esperados={LABELS}")

    x_train_text = train_df["resp_text"].tolist()
    x_test_text = test_df["resp_text"].tolist()

    log(f"Treino carregado: {len(train_df):,} exemplos")
    log(f"Teste carregado: {len(test_df):,} exemplos")
    print("Distribuição das classes no treino:")
    print(train_df["clarity"].value_counts().reindex(LABELS))

    return train_df, test_df, x_train_text, y, x_test_text


# -----------------------------------------------------------------------------
# SPLADE-PT-BR congelado -> scipy.sparse.csr_matrix float32
# -----------------------------------------------------------------------------
def import_splade_class(model_id: str = MODEL_ID, revision: str = MODEL_REVISION):
    """
    Baixa apenas o arquivo de definição do próprio repositório do modelo e importa
    a classe Splade diretamente. Evita depender de um checkout separado do pacote
    SPLADE e mantém a arquitetura usada pelo checkpoint.
    """
    log(f"Obtendo definição do SPLADE em {model_id}@{revision[:8]}")
    modeling_path = hf_hub_download(
        repo_id=model_id,
        filename="modeling_splade.py",
        revision=revision,
    )
    spec = importlib.util.spec_from_file_location("splade_ptbr_modeling", modeling_path)
    if spec is None or spec.loader is None:
        raise RuntimeError("Não foi possível importar modeling_splade.py")
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    return module.Splade


def load_splade_encoder(
    model_id: str = MODEL_ID,
    revision: str = MODEL_REVISION,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    Splade = import_splade_class(model_id, revision)

    # O construtor original chama AutoModelForMaskedLM.from_pretrained().
    # Quando o carregamento externo cria o SPLADE em meta, essa segunda chamada
    # falha. Aqui só criamos a arquitetura e depois atribuimos o checkpoint.
    class SpladeFromConfig(Splade):
        def __init__(self, config):
            PreTrainedModel.__init__(self, config)
            self.transformer = AutoModelForMaskedLM.from_config(config)
            self.aggregation = getattr(config, "aggregation", "max")
            self.fp16 = getattr(config, "fp16", True)

    torch_version = tuple(int(part) for part in torch.__version__.split("+")[0].split(".")[:2])
    if torch_version < (2, 6):
        raise RuntimeError("Este carregador requer PyTorch >= 2.6. Atualize e reinicie o kernel.")

    config_path = hf_hub_download(
        repo_id=model_id, filename="config.json", revision=revision
    )
    with open(config_path, encoding="utf-8") as file:
        config_dict = json.load(file)
    if config_dict.get("model_type") != "bert":
        raise ValueError("Este carregador espera a configuração BERT da revisão fixa do SPLADE-PT-BR.")
    config = BertConfig.from_dict(config_dict)

    log(f"Carregando tokenizer: {model_id}")
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)

    log("Criando arquitetura SPLADE sem alocar uma cópia inicial dos pesos")
    # Apenas parâmetros ficam em meta; buffers como position_ids ficam na CPU.
    # Nenhum from_pretrained é chamado dentro deste contexto.
    with torch.device("cpu"), init_empty_weights(include_buffers=False):
        model = SpladeFromConfig(config)

    log(f"Carregando checkpoint SPLADE com mmap: {model_id}@{revision[:8]}")
    weights_path = hf_hub_download(
        repo_id=model_id, filename="pytorch_model.bin", revision=revision
    )
    state_dict = torch.load(
        weights_path, map_location="cpu", weights_only=True, mmap=True
    )
    # Suporta tanto o checkpoint BERT MLM quanto o wrapper oficial SPLADE.
    # strict=True impede usar pesos aleatórios ou ignorar camadas incompatíveis.
    if state_dict and all(key.startswith("transformer.") for key in state_dict):
        model.load_state_dict(state_dict, strict=True, assign=True)
        log("Pesos do wrapper SPLADE carregados: todas as chaves conferidas")
    elif state_dict and all(key.startswith(("bert.", "cls.")) for key in state_dict):
        model.transformer.load_state_dict(state_dict, strict=True, assign=True)
        log("Pesos BERT MLM do checkpoint SPLADE carregados: todas as chaves conferidas")
    else:
        raise RuntimeError("Formato de chaves inesperado no checkpoint SPLADE; carregamento interrompido.")
    del state_dict
    model.transformer.tie_weights()

    meta_tensors = [
        name for name, tensor in list(model.named_parameters()) + list(model.named_buffers())
        if tensor.is_meta
    ]
    if meta_tensors:
        raise RuntimeError(f"Checkpoint incompleto: tensores ainda em meta: {meta_tensors[:5]}")
    model.eval()
    model.to(device)

    for parameter in model.parameters():
        parameter.requires_grad_(False)

    log(f"SPLADE pronto em {device}. Fine-tuning desativado.")
    return tokenizer, model, device


def _fingerprint_texts(texts: list[str], prefix: str) -> str:
    digest = hashlib.sha256()
    digest.update(MODEL_ID.encode("utf-8"))
    digest.update(MODEL_REVISION.encode("utf-8"))
    digest.update(ENCODER_CACHE_VERSION.encode("ascii"))
    digest.update(str(MAX_LENGTH).encode("ascii"))
    digest.update(prefix.encode("utf-8"))
    for text in texts:
        digest.update(text.encode("utf-8", errors="replace"))
        digest.update(b"\0")
    return digest.hexdigest()[:16]


def cache_path_for(texts: list[str], prefix: str) -> Path:
    return CACHE_DIR / f"{prefix}_{_fingerprint_texts(texts, prefix)}.npz"


def encode_splade_sparse(
    texts: list[str],
    tokenizer,
    model,
    device: torch.device,
    *,
    batch_size: int = BATCH_SIZE,
    max_length: int = MAX_LENGTH,
    description: str = "SPLADE",
) -> sparse.csr_matrix:
    """
    Codifica lotes pequenos. Apenas o lote corrente existe como tensor denso;
    o conjunto completo é montado em CSR float32.
    """
    blocks: list[sparse.csr_matrix] = []
    use_amp = device.type == "cuda"
    amp_context: Callable[[], object]
    amp_context = (
        lambda: torch.autocast(device_type="cuda", dtype=torch.float16)
        if use_amp
        else nullcontext()
    )

    model.eval()
    starts = range(0, len(texts), batch_size)
    progress = tqdm(starts, total=(len(texts) + batch_size - 1) // batch_size, desc=description)

    with torch.inference_mode():
        for batch_number, start in enumerate(progress, start=1):
            batch_texts = texts[start : start + batch_size]
            tokens = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            tokens = {name: tensor.to(device, non_blocking=True) for name, tensor in tokens.items()}

            with amp_context():
                # As respostas são tratadas como documentos, portanto usamos d_rep.
                dense_rep = model(d_kwargs=tokens)["d_rep"]

            batch_np = dense_rep.detach().float().cpu().numpy().astype(np.float32, copy=False)
            batch_csr = sparse.csr_matrix(batch_np, dtype=np.float32)
            batch_csr.eliminate_zeros()
            blocks.append(batch_csr)

            del tokens, dense_rep, batch_np, batch_csr
            if device.type == "cuda" and batch_number % 25 == 0:
                torch.cuda.empty_cache()

    if not blocks:
        return sparse.csr_matrix((0, 0), dtype=np.float32)

    matrix = sparse.vstack(blocks, format="csr", dtype=np.float32)
    matrix.sort_indices()
    return matrix


def get_or_create_representation(
    texts: list[str],
    prefix: str,
    tokenizer=None,
    model=None,
    device: torch.device | None = None,
) -> sparse.csr_matrix:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cache_path = cache_path_for(texts, prefix)

    if USE_CACHE and cache_path.exists():
        log(f"Carregando representação em cache: {cache_path}")
        matrix = sparse.load_npz(cache_path).tocsr().astype(np.float32, copy=False)
        log_sparse_matrix(prefix, matrix)
        return matrix

    if tokenizer is None or model is None or device is None:
        raise ValueError("Tokenizer/model/device são necessários quando não há cache")

    log(f"Gerando representação SPLADE para '{prefix}'")
    matrix = encode_splade_sparse(
        texts,
        tokenizer,
        model,
        device,
        description=f"SPLADE {prefix}",
    )
    log_sparse_matrix(prefix, matrix)

    if USE_CACHE:
        sparse.save_npz(cache_path, matrix, compressed=True)
        log(f"Cache salvo: {cache_path}")

    return matrix


def build_representations(
    x_train_text: list[str],
    x_test_text: list[str],
) -> tuple[sparse.csr_matrix, sparse.csr_matrix]:
    train_cache = cache_path_for(x_train_text, "train")
    test_cache = cache_path_for(x_test_text, "test")

    tokenizer = model = device = None
    if not (USE_CACHE and train_cache.exists() and test_cache.exists()):
        tokenizer, model, device = load_splade_encoder()

    try:
        x_train = get_or_create_representation(
            x_train_text, "train", tokenizer, model, device
        )
        x_test = get_or_create_representation(
            x_test_text, "test", tokenizer, model, device
        )
    finally:
        if model is not None:
            del model
        if tokenizer is not None:
            del tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        log("Codificador SPLADE liberado da memória")

    if x_train.shape[1] != x_test.shape[1]:
        raise RuntimeError("Treino e teste ficaram com números diferentes de características")

    return x_train, x_test


# -----------------------------------------------------------------------------
# Classificadores sklearn adequados à representação esparsa
# -----------------------------------------------------------------------------






def save_cv_artifacts(result: dict, y_true: np.ndarray) -> None:
    ACTIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    slug = result["model_name"].lower().replace(" ", "_")

    result["folds"].to_csv(ACTIVE_RESULTS_DIR / f"folds_{slug}.csv", index=False)
    result["report"].to_csv(ACTIVE_RESULTS_DIR / f"metricas_classes_{slug}.csv")

    cm_df = pd.DataFrame(result["confusion_matrix"], index=LABELS, columns=LABELS)
    cm_df.to_csv(ACTIVE_RESULTS_DIR / f"matriz_confusao_{slug}.csv")

    oof_df = pd.DataFrame(
        {
            "row_id": np.arange(len(y_true)),
            "fold": result["oof_fold"],
            "clarity_real": y_true,
            "clarity_pred": result["oof_pred"],
            "score_c1": result["oof_scores"][:, 0],
            "score_c234": result["oof_scores"][:, 1],
            "score_c5": result["oof_scores"][:, 2],
        }
    )
    oof_df.to_csv(ACTIVE_RESULTS_DIR / f"oof_{slug}.csv", index=False)
    log(f"Artefatos OOF/metrics salvos para {result['model_name']}")




# -----------------------------------------------------------------------------
# Relatórios, análise curta e comparação com experimentos anteriores
# -----------------------------------------------------------------------------
def model_summary(results: dict[str, dict]) -> pd.DataFrame:
    rows = []
    for name, result in results.items():
        rows.append(
            {
                "modelo": name,
                "acuracia_media": result["mean_accuracy"],
                "desvio_acuracia": result["std_accuracy"],
                "acuracia_treino_media": result["mean_train_accuracy"],
                "gap_medio": result["mean_gap"],
                "f1_macro_oof": float(result["report"].loc["macro avg", "f1-score"]),
            }
        )
    return pd.DataFrame(rows).sort_values("acuracia_media", ascending=False).reset_index(drop=True)


def normalized_confusion(cm: np.ndarray) -> pd.DataFrame:
    row_sums = cm.sum(axis=1, keepdims=True)
    normalized = np.divide(
        cm,
        row_sums,
        out=np.zeros_like(cm, dtype=np.float64),
        where=row_sums != 0,
    )
    return pd.DataFrame(normalized, index=LABELS, columns=LABELS)


def short_behavior_analysis(result: dict) -> str:
    cm = result["confusion_matrix"]
    row_sums = cm.sum(axis=1)
    recalls = np.divide(
        np.diag(cm),
        row_sums,
        out=np.zeros(len(LABELS), dtype=float),
        where=row_sums != 0,
    )

    best_idx = int(np.argmax(recalls))
    worst_idx = int(np.argmin(recalls))

    offdiag = cm.copy()
    np.fill_diagonal(offdiag, 0)
    src_idx, dst_idx = np.unravel_index(np.argmax(offdiag), offdiag.shape)
    largest_error = int(offdiag[src_idx, dst_idx])
    largest_error_rate = (
        largest_error / row_sums[src_idx] if row_sums[src_idx] else 0.0
    )

    c234_idx = LABELS.index("c234")
    c234_total = row_sums[c234_idx]
    c234_to_c1 = cm[c234_idx, LABELS.index("c1")]
    c234_to_c5 = cm[c234_idx, LABELS.index("c5")]

    lines = [
        f"Classe com maior recall: {LABELS[best_idx]} ({recalls[best_idx]:.2%}).",
        f"Classe com menor recall: {LABELS[worst_idx]} ({recalls[worst_idx]:.2%}).",
        (
            f"Confusão mais frequente: {LABELS[src_idx]} → {LABELS[dst_idx]}: "
            f"{largest_error} exemplos ({largest_error_rate:.2%} da classe de origem)."
        ),
        (
            f"c234: {int(c234_to_c1)} erros para c1 e {int(c234_to_c5)} erros para c5 "
            f"em {int(c234_total)} exemplos."
        ),
        (
            f"Gap médio treino-validação: {result['mean_gap']:.2%}. "
            f"Desvio da acurácia entre folds: {result['std_accuracy']:.2%}."
        ),
    ]
    return "\n".join(lines)






def print_detailed_results(results: dict[str, dict]) -> pd.DataFrame:
    for name, result in results.items():
        print("\n" + "=" * 90)
        print(name)
        print("=" * 90)
        print("\nResultados dos 10 folds:")
        print(
            result["folds"][
                [
                    "fold",
                    "acuracia_treino",
                    "acuracia_validacao",
                    "gap_treino_validacao",
                    "f1_macro_validacao",
                    "tempo_s",
                ]
            ].to_string(index=False)
        )
        print(
            f"\nAcurácia média: {result['mean_accuracy']:.2%} "
            f"± {result['std_accuracy']:.2%}"
        )
        print(f"Acurácia média de treino: {result['mean_train_accuracy']:.2%}")
        print(f"Gap médio treino-validação: {result['mean_gap']:.2%}")

        print("\nMétricas por classe (OOF agregado):")
        rows = LABELS + ["macro avg", "weighted avg"]
        print(result["report"].loc[rows].to_string())

        cm_df = pd.DataFrame(result["confusion_matrix"], index=LABELS, columns=LABELS)
        print("\nMatriz de confusão agregada (linhas=reais, colunas=preditas):")
        print(cm_df.to_string())
        print("\nMatriz de confusão normalizada por classe real:")
        print(normalized_confusion(result["confusion_matrix"]).to_string(float_format=lambda x: f"{x:.2%}"))

        print("\nAnálise curta:")
        print(short_behavior_analysis(result))

    summary = model_summary(results)
    print("\n" + "=" * 90)
    print("Resumo dos classificadores")
    print("=" * 90)
    print(summary.to_string(index=False))

    comparison = compare_with_previous(summary)
    print("\nComparação com resultados anteriores:")
    print(comparison.to_string(index=False))

    print("\n" + approach_conclusion(summary))
    return summary


# -----------------------------------------------------------------------------
# Treino final e rotulação do conjunto de teste
# -----------------------------------------------------------------------------




# -----------------------------------------------------------------------------
# Execução completa pelo arquivo .py
# -----------------------------------------------------------------------------
def main() -> None:
    log_versions()
    train_df, test_df, x_train_text, y, x_test_text = load_datasets()
    x_train, x_test = build_representations(x_train_text, x_test_text)

    results = run_cross_validation(x_train, y)
    summary = print_detailed_results(results)
    save_summary_json(summary, results)

    # A rotulação é gerada mesmo que a conclusão seja negativa, para permitir
    # inspeção; não implica que o modelo deva ser escolhido para a entrega final.
    train_final_and_predict(summary, x_train, y, x_test, test_df)

import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import Normalizer

INNER_SPLITS = 3
SEARCH_CANDIDATES = [
    {"normalizacao": "nenhuma", "C": c} for c in (0.001, 0.01, 0.1, 1.0)
] + [
    {"normalizacao": "l2", "C": c} for c in (0.01, 0.1, 1.0, 10.0, 100.0)
]
REFERENCE_PARAMS = {"normalizacao": "nenhuma", "C": 1.0}
REFERENCE_NAME = "LR_referencia_C1"
TUNED_NAME = "LR_busca_generalizacao"
RESUME_FOLDS = True


def make_lr_pipeline(params):
    return Pipeline([
        ("normalizacao", Normalizer(norm="l2", copy=True)
         if params["normalizacao"] == "l2" else "passthrough"),
        ("clf", LogisticRegression(
            C=float(params["C"]), class_weight="balanced", solver="saga",
            max_iter=1000, tol=1e-3, random_state=RANDOM_STATE,
        )),
    ])


def fit_checked(clf, x, y):
    # Mesmos solver/tolerância da V4; amplia iterações somente se necessário.
    for limit in (1000, 3000, 5000):
        clf.set_params(clf__max_iter=limit)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            clf.fit(x, y)
        convergence_warnings = [w for w in caught if issubclass(w.category, ConvergenceWarning)]
        for w in caught:
            if not issubclass(w.category, ConvergenceWarning):
                warnings.warn(str(w.message), w.category)
        if not convergence_warnings:
            return {"n_iter": int(np.max(clf.named_steps["clf"].n_iter_)),
                    "max_iter": limit, "convergiu": True}
        log(f"Convergência não atingida em {limit} iterações; ampliando limite")
    raise RuntimeError("Classificador não convergiu em 5000 iterações; configuração rejeitada.")


def experiment_signature(x, y):
    h = hashlib.sha256()
    settings = {"code": CODE_VERSION, "shape": x.shape, "outer": N_SPLITS,
                "inner": INNER_SPLITS, "seed": RANDOM_STATE, "grid": SEARCH_CANDIDATES,
                "reference": REFERENCE_PARAMS, "dependencies": REQUIRED_VERSIONS,
                "solver": "saga", "tol": 1e-3, "class_weight": "balanced"}
    h.update(json.dumps(settings, sort_keys=True).encode())
    for array in (x.data, x.indices, x.indptr):
        h.update(str(array.dtype).encode())
        h.update(np.ascontiguousarray(array).tobytes())
    h.update(json.dumps(np.asarray(y).astype(str).tolist()).encode())
    return h.hexdigest()[:16]


def atomic_json(path, payload):
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    temp.replace(path)


def search_regularization(x, y, *, seed, context, output_csv):
    inner_cv = StratifiedKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=seed)
    # Os mesmos folds internos são usados por todas as configurações.
    splits = list(inner_cv.split(np.zeros(len(y)), y))
    rows = []
    for candidate_idx, params in enumerate(SEARCH_CANDIDATES, start=1):
        val_scores, train_scores, iterations = [], [], []
        started = time.time()
        error = ""
        log(f"{context} | configuração {candidate_idx}/{len(SEARCH_CANDIDATES)} | {params}")
        for inner_fold, (ti, vi) in enumerate(splits, start=1):
            clf = make_lr_pipeline(params)
            try:
                stats = fit_checked(clf, x[ti], y[ti])
            except RuntimeError as exc:
                error = str(exc)
                log(f"{context} | configuração rejeitada: {error}")
                break
            train_scores.append(float(accuracy_score(y[ti], clf.predict(x[ti]))))
            val_scores.append(float(accuracy_score(y[vi], clf.predict(x[vi]))))
            iterations.append(stats["n_iter"])
            log(f"{context} | interno {inner_fold}/{INNER_SPLITS} | "
                f"treino={train_scores[-1]:.2%} | val={val_scores[-1]:.2%} | "
                f"iterações={stats['n_iter']}")
            del clf
        valid = len(val_scores) == INNER_SPLITS and not error
        rows.append({"configuracao": candidate_idx, **params,
                     "acuracia_interna": float(np.mean(val_scores)) if valid else None,
                     "acuracia_treino_interna": float(np.mean(train_scores)) if valid else None,
                     "desvio_interno": float(np.std(val_scores, ddof=1)) if valid else None,
                     "iteracoes_max": max(iterations) if iterations else None,
                     "valida": valid, "erro": error, "tempo_s": time.time() - started})
        pd.DataFrame(rows).to_csv(output_csv, index=False)
        gc.collect()
    valid_rows = [r for r in rows if r["valida"]]
    if not valid_rows:
        raise RuntimeError("Nenhuma configuração convergiu; veja o CSV da busca.")
    # Seleção por acurácia interna. Empate: menor C, seguido de ordem estável.
    best = sorted(valid_rows, key=lambda r: (-r["acuracia_interna"], r["C"], r["configuracao"]))[0]
    params = {"normalizacao": best["normalizacao"], "C": best["C"]}
    log(f"{context} | escolhido SOMENTE na CV interna: {params} | "
        f"acurácia interna={best['acuracia_interna']:.2%}")
    return params, pd.DataFrame(rows)


def pack_oof_result(name, rows, pred, fold_ids, scores, y):
    if (fold_ids == 0).any() or not np.isfinite(scores).all():
        raise RuntimeError("OOF incompleto; a avaliação foi interrompida.")
    frame = pd.DataFrame(rows)
    report = pd.DataFrame(classification_report(
        y, pred, labels=LABELS, output_dict=True, zero_division=0)).T
    return {"model_name": name, "folds": frame, "oof_pred": pred,
            "oof_fold": fold_ids, "oof_scores": scores, "score_type": "probability",
            "mean_accuracy": float(frame.acuracia_validacao.mean()),
            "std_accuracy": float(frame.acuracia_validacao.std(ddof=1)),
            "mean_train_accuracy": float(frame.acuracia_treino.mean()),
            "mean_gap": float(frame.gap_treino_validacao.mean()),
            "report": report, "confusion_matrix": confusion_matrix(y, pred, labels=LABELS)}


def run_cross_validation(x_train, y):
    global ACTIVE_RESULTS_DIR
    if not sparse.isspmatrix_csr(x_train):
        raise TypeError("A representação deve permanecer CSR.")
    if not np.isfinite(x_train.data).all():
        raise ValueError("A matriz contém valores não finitos.")
    y = np.asarray(y)
    signature = experiment_signature(x_train, y)
    ACTIVE_RESULTS_DIR = RESULTS_DIR / signature
    ACTIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    log(f"Resultados e checkpoints: {ACTIVE_RESULTS_DIR}")
    splits = list(StratifiedKFold(n_splits=N_SPLITS, shuffle=True,
                                 random_state=RANDOM_STATE).split(np.zeros(len(y)), y))
    names = [REFERENCE_NAME, TUNED_NAME]
    accumulator = {name: {"rows": [], "pred": np.empty(len(y), dtype=object),
                          "fold": np.zeros(len(y), dtype=np.int16),
                          "scores": np.full((len(y), len(LABELS)), np.nan, dtype=np.float32)}
                   for name in names}
    for fold, (ti, vi) in enumerate(tqdm(splits, desc="CV externa"), start=1):
        checkpoint = ACTIVE_RESULTS_DIR / f"fold_{fold:02d}.json"
        if RESUME_FOLDS and checkpoint.exists():
            saved = json.loads(checkpoint.read_text())
            if saved["signature"] != signature or saved["validation_indices"] != vi.tolist():
                raise RuntimeError("Checkpoint incompatível com os dados/folds.")
            log(f"Fold externo {fold}/{N_SPLITS}: retomando resultado concluído")
        else:
            outer_started = time.time()
            log(f"Fold externo {fold}/{N_SPLITS} | treino={len(ti)} | val={len(vi)}")
            # A busca recebe SOMENTE as linhas de treino do fold externo.
            params, _ = search_regularization(
                x_train[ti], y[ti], seed=RANDOM_STATE + fold, context=f"Externo {fold}",
                output_csv=ACTIVE_RESULTS_DIR / f"busca_interna_fold_{fold:02d}.csv")
            saved = {"signature": signature, "validation_indices": vi.tolist(), "models": {}}
            for name, current_params in [(REFERENCE_NAME, REFERENCE_PARAMS), (TUNED_NAME, params)]:
                start = time.time()
                clf = make_lr_pipeline(current_params)
                stats = fit_checked(clf, x_train[ti], y[ti])
                train_acc = float(accuracy_score(y[ti], clf.predict(x_train[ti])))
                val_pred = clf.predict(x_train[vi])
                val_acc = float(accuracy_score(y[vi], val_pred))
                probabilities = clf.predict_proba(x_train[vi])
                class_order = [list(clf.classes_).index(label) for label in LABELS]
                aligned = probabilities[:, class_order].astype(np.float32)
                row = {"modelo": name, "fold": fold, "n_treino": len(ti), "n_validacao": len(vi),
                       "acuracia_treino": train_acc, "acuracia_validacao": val_acc,
                       "gap_treino_validacao": train_acc - val_acc,
                       "f1_macro_validacao": float(f1_score(y[vi], val_pred, labels=LABELS, average="macro")),
                       "tempo_s": time.time() - start, **current_params, **stats}
                saved["models"][name] = {"row": row, "pred": val_pred.tolist(), "scores": aligned.tolist()}
                log(f"{name} | fold {fold} | treino={train_acc:.2%} | val={val_acc:.2%} | "
                    f"gap={(train_acc-val_acc)*100:.2f} p.p. | {current_params}")
                del clf
            saved["tempo_fold_completo_s"] = time.time() - outer_started
            saved["models"][TUNED_NAME]["row"]["tempo_s"] = saved["tempo_fold_completo_s"]
            atomic_json(checkpoint, saved)
        for name in names:
            part = saved["models"][name]
            accumulator[name]["rows"].append(part["row"])
            accumulator[name]["pred"][vi] = np.asarray(part["pred"], dtype=object)
            accumulator[name]["fold"][vi] = fold
            accumulator[name]["scores"][vi] = np.asarray(part["scores"], dtype=np.float32)
        gc.collect()
    results = {}
    for name, a in accumulator.items():
        results[name] = pack_oof_result(name, a["rows"], a["pred"], a["fold"], a["scores"], y)
        save_cv_artifacts(results[name], y)
    paired = pd.DataFrame({"fold": np.arange(1, N_SPLITS + 1),
                           "referencia": results[REFERENCE_NAME]["folds"].acuracia_validacao,
                           "busca": results[TUNED_NAME]["folds"].acuracia_validacao})
    paired["ganho_pp"] = (paired.busca - paired.referencia) * 100
    paired.to_csv(ACTIVE_RESULTS_DIR / "comparacao_pareada.csv", index=False)
    log(f"Ganho médio pareado da busca: {paired.ganho_pp.mean():+.2f} p.p.; "
        f"melhora em {int((paired.ganho_pp > 0).sum())}/{N_SPLITS} folds")
    return results


def compare_with_previous(summary_df):
    current = summary_df[["modelo", "acuracia_media"]].copy()
    current["folds"] = N_SPLITS
    current["comparacao_direta"] = True
    current["observacao"] = "Mesmos folds externos nesta execução; busca interna isolada"
    previous = pd.DataFrame(PREVIOUS_RESULTS)
    previous["comparacao_direta"] = False
    comparison = pd.concat([current, previous], ignore_index=True)
    comparison["delta_vs_baseline_historico_pp"] = (comparison.acuracia_media - 0.4565) * 100
    return comparison


def approach_conclusion(summary_df):
    keyed = summary_df.set_index("modelo")
    tuned = keyed.loc[TUNED_NAME]
    reference = keyed.loc[REFERENCE_NAME]
    delta = (tuned.acuracia_media - reference.acuracia_media) * 100
    gap_delta = (tuned.gap_medio - reference.gap_medio) * 100
    return (f"Busca avaliada nos folds externos: {tuned.acuracia_media:.2%}. "
            f"Mudança de acurácia frente à referência: {delta:+.2f} p.p.; "
            f"mudança do gap: {gap_delta:+.2f} p.p. "
            "Reduzir o gap só é útil se a validação também melhorar. "
            "O baseline histórico de 45,65% não foi reexecutado aqui.")


def save_summary_json(summary_df, results):
    payload = {"code_version": CODE_VERSION, "representation": MODEL_ID,
               "model_revision": MODEL_REVISION, "encoder_cache_version": ENCODER_CACHE_VERSION,
               "max_length": MAX_LENGTH, "n_splits": N_SPLITS, "inner_splits": INNER_SPLITS,
               "random_state": RANDOM_STATE, "candidates": SEARCH_CANDIDATES,
               "dependencies": validate_environment(), "models": {},
               "final_procedure": TUNED_NAME,
               "selection": "acurácia interna; empate por menor C",
               "historical_baseline_reexecuted": False}
    for name, result in results.items():
        payload["models"][name] = {"mean_accuracy": result["mean_accuracy"],
                                   "std_accuracy": result["std_accuracy"],
                                   "mean_train_accuracy": result["mean_train_accuracy"],
                                   "mean_gap": result["mean_gap"],
                                   "macro_f1_oof": float(result["report"].loc["macro avg", "f1-score"]),
                                   "folds": result["folds"].to_dict(orient="records")}
    atomic_json(ACTIVE_RESULTS_DIR / "resumo_experimento.json", payload)


def train_final_and_predict(summary_df, x_train, y, x_test, test_df):
    # Procedimento pré-definido: nova busca interna em TODO o treino rotulado.
    # Não usa os scores externos para escolher C ou normalização.
    params, _ = search_regularization(
        x_train, np.asarray(y), seed=RANDOM_STATE + 999, context="Busca final",
        output_csv=ACTIVE_RESULTS_DIR / "busca_final.csv")
    clf = make_lr_pipeline(params)
    stats = fit_checked(clf, x_train, np.asarray(y))
    pred = clf.predict(x_test)
    if not set(pred).issubset(set(LABELS)):
        raise RuntimeError("Rótulo inválido.")
    output = pd.DataFrame({"resp_text": test_df.resp_text.to_numpy(), "clarity": pred})
    if len(output) != len(test_df):
        raise RuntimeError("Número de previsões incompatível.")
    output.to_excel(ACTIVE_RESULTS_DIR / "teste_rotulado_splade.xlsx", index=False)
    atomic_json(ACTIVE_RESULTS_DIR / "configuracao_final.json", {"params": params, **stats})
    log(f"Modelo final: {params}; arquivo salvo com {len(output)} respostas")
    return clf, output


def diagnose_representations(x_train, x_test):
    for name, x in [("Treino", x_train), ("Teste", x_test)]:
        norms = np.sqrt(np.asarray(x.multiply(x).sum(axis=1)).ravel())
        print(f"{name}: linhas nulas={int((x.getnnz(axis=1)==0).sum())}; "
              f"norma L2 p10/mediana/p90={np.quantile(norms, [0.1,0.5,0.9])}")


def diagnose_truncation(texts):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
    lengths = []
    for start in tqdm(range(0, len(texts), 128), desc="Diagnóstico de truncamento"):
        batch = tokenizer(texts[start:start+128], padding=False, truncation=False,
                          return_length=True, verbose=False)
        lengths.extend(batch["length"])
    lengths = np.asarray(lengths)
    stats = {"n_respostas": len(texts), "max_length": MAX_LENGTH,
             "acima_do_limite": int((lengths > MAX_LENGTH).sum()),
             "proporcao_truncada": float((lengths > MAX_LENGTH).mean()),
             "mediana_tokens": float(np.median(lengths)),
             "p90_tokens": float(np.quantile(lengths, 0.9))}
    print(json.dumps(stats, ensure_ascii=False, indent=2))
    return stats


from IPython.display import display

log_versions()
print(f"Representação: {MODEL_ID}@{MODEL_REVISION[:8]}")
print(f"CV: {N_SPLITS} folds | Batch size: {BATCH_SIZE} | Max length: {MAX_LENGTH}")

print(f"Busca: {len(SEARCH_CANDIDATES)} configurações | CV interna: {INNER_SPLITS} folds | solver: CPU")


Código ativo: SPLADE_EP1_COLAB_V5_GENERALIZACAO
Python: 3.13.15
  torch           : 2.11.0+cu130 [versão verificada]
  transformers    : 4.57.3 [versão verificada]
  accelerate      : 1.15.0 [versão verificada]
  huggingface_hub : 0.36.2 [versão verificada]
  tokenizers      : 0.22.2 [versão verificada]
  safetensors     : 0.8.0 [versão verificada]
  numpy           : 2.3.5 [versão verificada]
  pandas          : 2.2.3 [versão verificada]
  scipy           : 1.17.0 [versão verificada]
  scikit-learn    : 1.8.0 [versão verificada]
  tqdm            : 4.70.1 [versão verificada]
  psutil          : 7.2.2 [versão verificada]
  openpyxl        : 3.1.5 [versão verificada]
  setuptools      : 81.0.0 [versão verificada]
Dispositivo: cuda
Representação: AxelPCG/splade-pt-br@fff5e6a3
CV: 10 folds | Batch size: 4 | Max length: 256
Busca: 9 configurações | CV interna: 3 folds | solver: CPU


## Importação dos conjuntos

Não há pré-processamento linguístico manual. O texto é preservado para o tokenizer do SPLADE-PT-BR; apenas valores ausentes são convertidos para string vazia. Isso evita inserir uma etapa de engenharia manual de características.

In [3]:
dados_treino, dados_teste, X_texto_treino, Y_treino, X_texto_teste = load_datasets()

display(dados_treino.head())

[13:02:03] Carregando treino: dados/train.xlsx | RSS≈0.90 GB
[13:02:06] Carregando teste: dados/test1.xlsx | RSS≈0.91 GB
[13:02:06] Treino carregado: 20,092 exemplos | RSS≈0.91 GB
[13:02:06] Teste carregado: 900 exemplos | RSS≈0.91 GB
Distribuição das classes no treino:
clarity
c1      6347
c234    6853
c5      6892
Name: count, dtype: int64


,resp_text,clarity
0,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c5
1,"Prezada cidadã, As informações sobre óbitos ...",c1
2,"Prezado Senhor Julio, A Ouvidoria-Geral da P...",c1
3,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c234
4,"Senhor, O Serviço de Informações ao Cidadão d...",c234


## Representação SPLADE-PT-BR

O encoder é carregado em modo de avaliação, com todos os parâmetros congelados. Cada resposta é codificada como `d_rep` (representação de documento). A saída de cada lote é imediatamente convertida para CSR `float32`.

Como o encoder é **pré-treinado e não é ajustado ao conjunto do EP**, a representação pode ser calculada uma única vez antes da validação cruzada sem usar rótulos nem aprender estatísticas dos folds. O classificador scikit-learn, por outro lado, é ajustado separadamente em cada fold.

O cache usa um hash dos textos + modelo + revisão + comprimento máximo, evitando reaproveitar uma matriz incompatível após alteração dos dados/configuração.
O carregador cria a arquitetura com `from_config()` e atribui diretamente os pesos SPLADE com `mmap=True`, `weights_only=True`, `assign=True` e `strict=True`. Não chama `Splade.from_pretrained()`. Todas as chaves dos pesos são conferidas e a inferência é bloqueada se restar algum tensor `meta`.


In [4]:
X_treino, X_teste = build_representations(X_texto_treino, X_texto_teste)

print(type(X_treino), X_treino.dtype, X_treino.shape)
print(type(X_teste), X_teste.dtype, X_teste.shape)

[13:02:07] Obtendo definição do SPLADE em AxelPCG/splade-pt-br@fff5e6a3 | RSS≈0.91 GB


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modeling_splade.py: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/797 [00:00<?, ?B/s]

[13:02:08] Carregando tokenizer: AxelPCG/splade-pt-br | RSS≈0.91 GB


...

[13:02:15] Pesos BERT MLM do checkpoint SPLADE carregados: todas as chaves conferidas | RSS≈1.05 GB
[13:02:15] SPLADE pronto em cuda. Fine-tuning desativado. | RSS≈1.14 GB
[13:02:15] Gerando representação SPLADE para 'train' | RSS≈1.14 GB


SPLADE train:   0%|          | 0/5023 [00:00<?, ?it/s]

[13:06:10] train: shape=(20092, 29794), nnz=8,603,655, esparsidade=98.5628%, CSR≈65.72 MB, denso float32≈2283.56 MB | RSS≈1.50 GB
[13:06:14] Cache salvo: cache_splade/train_8a777c7854daaeb8.npz | RSS≈1.54 GB
[13:06:14] Gerando representação SPLADE para 'test' | RSS≈1.54 GB


SPLADE test:   0%|          | 0/225 [00:00<?, ?it/s]

[13:06:25] test: shape=(900, 29794), nnz=386,757, esparsidade=98.5577%, CSR≈2.95 MB, denso float32≈102.29 MB | RSS≈1.54 GB
[13:06:25] Cache salvo: cache_splade/test_8468f7812b8323ad.npz | RSS≈1.54 GB
[13:06:26] Codificador SPLADE liberado da memória | RSS≈1.54 GB
<class 'scipy.sparse._csr.csr_matrix'> float32 (20092, 29794)
<class 'scipy.sparse._csr.csr_matrix'> float32 (900, 29794)


## Diagnóstico da representação

As normas ajudam a verificar a magnitude dos vetores. O diagnóstico de truncamento é opcional e usa apenas o tokenizer na CPU; não altera as representações nem entra no grid.


In [5]:
diagnose_representations(X_treino, X_teste)

EXECUTAR_DIAGNOSTICO_TRUNCAMENTO = False
if EXECUTAR_DIAGNOSTICO_TRUNCAMENTO:
    diagnostico_truncamento = diagnose_truncation(X_texto_treino)


Treino: linhas nulas=0; norma L2 p10/mediana/p90=[ 8.69650545 11.18271399 13.02916279]
Teste: linhas nulas=0; norma L2 p10/mediana/p90=[ 8.58746719 11.23002052 13.13922577]


## Regressão logística e controle do overfitting

Comparamos o procedimento de busca com a referência `C=1`, sem normalização. Ambos usam a mesma representação e as mesmas divisões externas. A busca escolhe pela acurácia interna; treino, gap e F1 são diagnósticos, não critérios de seleção.

Convergência é registrada. Se necessário, o limite de iterações é ampliado de 1000 para 3000 e 5000; configurações que ainda não convergirem são rejeitadas.


## Validação externa em 10 folds

Os rótulos de validação externos não participam da busca. As pontuações internas são usadas para seleção; as pontuações externas e as previsões OOF são usadas para avaliar generalização. Resultados de cada fold são salvos antes de avançar.


In [6]:
resultados = run_cross_validation(X_treino, Y_treino)

[13:06:26] Resultados e checkpoints: resultados_splade_v5/b76882741c1392a8 | RSS≈1.54 GB


CV externa:   0%|          | 0/10 [00:00<?, ?it/s]

[13:06:26] Fold externo 1/10 | treino=18082 | val=2010 | RSS≈1.54 GB
[13:06:26] Externo 1 | configuração 1/9 | {'normalizacao': 'nenhuma', 'C': 0.001} | RSS≈1.57 GB
[13:06:29] Externo 1 | interno 1/3 | treino=49.97% | val=44.77% | iterações=13 | RSS≈1.60 GB
[13:06:32] Externo 1 | interno 2/3 | treino=49.28% | val=44.68% | iterações=13 | RSS≈1.60 GB
[13:06:34] Externo 1 | interno 3/3 | treino=49.57% | val=44.13% | iterações=13 | RSS≈1.60 GB
[13:06:34] Externo 1 | configuração 2/9 | {'normalizacao': 'nenhuma', 'C': 0.01} | RSS≈1.60 GB
[13:06:37] Externo 1 | interno 1/3 | treino=59.99% | val=45.07% | iterações=18 | RSS≈1.60 GB
[13:06:39] Externo 1 | interno 2/3 | treino=60.49% | val=44.05% | iterações=17 | RSS≈1.60 GB
[13:06:43] Externo 1 | interno 3/3 | treino=60.30% | val=43.52% | iterações=17 | RSS≈1.60 GB
[13:06:43] Externo 1 | configuração 3/9 | {'normalizacao': 'nenhuma', 'C': 0.1} | RSS≈1.60 GB
[13:06:56] Externo 1 | interno 1/3 | treino=75.34% | val=43.12% | iterações=80 | RSS≈1.6

## Resultados, comparação pareada e previsões OOF

Acurácia, F1 por classe, matrizes de confusão e gap são calculados com as previsões externas. A comparação com o controle usa os mesmos folds. Não trate o melhor resultado da CV interna como acurácia de generalização.


In [7]:
resumo = print_detailed_results(resultados)
resumo


LR_referencia_C1

Resultados dos 10 folds:
 fold  acuracia_treino  acuracia_validacao  gap_treino_validacao  f1_macro_validacao   tempo_s
    1         0.818383            0.416418              0.401965            0.415464 65.979167
    2         0.820153            0.421393              0.398760            0.420177 68.649879
    3         0.818891            0.439024              0.379866            0.437771 70.851364
    4         0.817840            0.421105              0.396735            0.420473 64.649072
    5         0.818559            0.439522              0.379037            0.438477 67.048088
    6         0.817619            0.412643              0.404976            0.412977 64.864103
    7         0.819886            0.432554              0.387333            0.432009 64.499132
    8         0.819333            0.434047              0.385286            0.433212 64.315267
    9         0.817232            0.407666              0.409566            0.407640 63.240752
   10 

,modelo,acuracia_media,desvio_acuracia,acuracia_treino_media,gap_medio,f1_macro_oof
0,LR_busca_generalizacao,0.446995,0.006139,0.562849,0.115855,0.441848
1,LR_referencia_C1,0.425742,0.011357,0.818441,0.392698,0.425118


## Inspeção compacta por classificador

In [8]:
for nome, resultado in resultados.items():
    print("\n" + "=" * 80)
    print(nome)
    print("=" * 80)

    print("\n10 folds:")
    display(resultado["folds"])

    print("Métricas por classe (OOF agregado):")
    display(resultado["report"].loc[LABELS + ["macro avg", "weighted avg"]])

    print("Matriz de confusão agregada:")
    display(pd.DataFrame(resultado["confusion_matrix"], index=LABELS, columns=LABELS))

    print("Matriz normalizada por classe real:")
    display(normalized_confusion(resultado["confusion_matrix"]))

    print("Análise curta:")
    print(short_behavior_analysis(resultado))


LR_referencia_C1

10 folds:


,modelo,fold,n_treino,n_validacao,acuracia_treino,acuracia_validacao,gap_treino_validacao,f1_macro_validacao,tempo_s,normalizacao,C,n_iter,max_iter,convergiu
0,LR_referencia_C1,1,18082,2010,0.818383,0.416418,0.401965,0.415464,65.979167,nenhuma,1.0,270,1000,True
1,LR_referencia_C1,2,18082,2010,0.820153,0.421393,0.398760,0.420177,68.649879,nenhuma,1.0,284,1000,True
2,LR_referencia_C1,3,18083,2009,0.818891,0.439024,0.379866,0.437771,70.851364,nenhuma,1.0,288,1000,True
3,LR_referencia_C1,4,18083,2009,0.817840,0.421105,0.396735,0.420473,64.649072,nenhuma,1.0,270,1000,True
4,LR_referencia_C1,5,18083,2009,0.818559,0.439522,0.379037,0.438477,67.048088,nenhuma,1.0,284,1000,True
5,LR_referencia_C1,6,18083,2009,0.817619,0.412643,0.404976,0.412977,64.864103,nenhuma,1.0,274,1000,True
6,LR_referencia_C1,7,18083,2009,0.819886,0.432554,0.387333,0.432009,64.499132,nenhuma,1.0,288,1000,True
7,LR_referencia_C1,8,18083,2009,0.819333,0.434047,0.385286,0.433212,64.315267,nenhuma,1.0,279,1000,True
8,LR_referencia_C1,9,18083,2009,0.817232,0.407666,0.409566,0.407640,63.240752,nenhuma,1.0,275,1000,True
9,LR_referencia_C1,10,18083,2009,0.816513,0.433051,0.383461,0.432865,63.202889,nenhuma,1.0,268,1000,True


Métricas por classe (OOF agregado):


,precision,recall,f1-score,support
c1,0.424785,0.474712,0.448363,6347.0
c234,0.387458,0.353422,0.369658,6853.0
c5,0.462211,0.452554,0.457331,6892.0
macro avg,0.424818,0.426896,0.425118,20092.0
weighted avg,0.424891,0.425742,0.424595,20092.0


Matriz de confusão agregada:


...

Matriz de confusão agregada:


,c1,c234,c5
c1,3290,1488,1569
c234,2389,2065,2399
c5,1613,1653,3626


Matriz normalizada por classe real:


,c1,c234,c5
c1,0.518355,0.234441,0.247203
c234,0.348606,0.301328,0.350066
c5,0.234039,0.239843,0.526117


Análise curta:
Classe com maior recall: c5 (52.61%).
Classe com menor recall: c234 (30.13%).
Confusão mais frequente: c234 → c5: 2399 exemplos (35.01% da classe de origem).
c234: 2389 erros para c1 e 2399 erros para c5 em 6853 exemplos.
Gap médio treino-validação: 11.59%. Desvio da acurácia entre folds: 0.61%.


## Referências históricas

O baseline TF-IDF + LR de 45,65% e os resultados anteriores não são reexecutados aqui; são referências contextuais. A comparação direta desta execução é entre o controle e a busca nos mesmos folds externos.


In [9]:
comparacao = compare_with_previous(resumo)
display(comparacao)
print(approach_conclusion(resumo))

,modelo,acuracia_media,folds,comparacao_direta,observacao,delta_vs_baseline_historico_pp
0,LR_busca_generalizacao,0.446995,10,True,Mesmos folds externos nesta execução; busca in...,-0.950513
1,LR_referencia_C1,0.425742,10,True,Mesmos folds externos nesta execução; busca in...,-3.075773
2,Baseline TF-IDF + Regressão Logística,0.456500,10,False,baseline anterior do projeto; treino médio ~69...,0.000000
3,SetFit + CosineSimilarityLoss + LR balanced,0.401200,3,False,protocolo de 3 folds; usar apenas como referên...,-5.530000


Busca avaliada nos folds externos: 44.70%. Mudança de acurácia frente à referência: +2.13 p.p.; mudança do gap: -27.68 p.p. Reduzir o gap só é útil se a validação também melhorar. O baseline histórico de 45,65% não foi reexecutado aqui.


## Persistência do resumo

Além dos CSVs de folds, métricas, matriz de confusão e OOF, salvamos um JSON com a configuração principal e o resumo do experimento.

In [10]:
save_summary_json(resumo, resultados)
print(f"Resumo salvo em {ACTIVE_RESULTS_DIR / 'resumo_experimento.json'}")


Resumo salvo em resultados_splade_v5/b76882741c1392a8/resumo_experimento.json


## Busca final e rotulação de test1.xlsx

Depois da avaliação externa, repetimos a busca de 3 folds internos em todo o treino rotulado e ajustamos a configuração escolhida em 100% do treino. Os parâmetros finais não são escolhidos pela pontuação externa. A rotulação usa esse procedimento previamente definido mesmo se o controle tiver melhor acurácia; o relatório mostra se a busca trouxe ganho.

As 900 respostas de teste não têm rótulos conhecidos: suas previsões não fornecem uma medida de acurácia.


In [11]:
modelo_final, teste_rotulado = train_final_and_predict(
    resumo,
    X_treino,
    Y_treino,
    X_teste,
    dados_teste,
)

display(teste_rotulado.head())
print(f"Linhas geradas: {len(teste_rotulado)}")

[14:08:49] Busca final | configuração 1/9 | {'normalizacao': 'nenhuma', 'C': 0.001} | RSS≈1.51 GB
[14:08:52] Busca final | interno 1/3 | treino=49.87% | val=45.10% | iterações=16 | RSS≈1.55 GB
[14:08:54] Busca final | interno 2/3 | treino=49.42% | val=44.60% | iterações=14 | RSS≈1.55 GB
[14:08:57] Busca final | interno 3/3 | treino=49.54% | val=43.35% | iterações=14 | RSS≈1.55 GB
[14:08:57] Busca final | configuração 2/9 | {'normalizacao': 'nenhuma', 'C': 0.01} | RSS≈1.55 GB
[14:09:01] Busca final | interno 1/3 | treino=59.54% | val=45.10% | iterações=19 | RSS≈1.55 GB
[14:09:04] Busca final | interno 2/3 | treino=59.66% | val=44.95% | iterações=17 | RSS≈1.55 GB
[14:09:07] Busca final | interno 3/3 | treino=59.60% | val=44.17% | iterações=18 | RSS≈1.55 GB
[14:09:07] Busca final | configuração 3/9 | {'normalizacao': 'nenhuma', 'C': 0.1} | RSS≈1.55 GB
[14:09:21] Busca final | interno 1/3 | treino=73.53% | val=43.07% | iterações=78 | RSS≈1.55 GB
[14:09:33] Busca final | interno 2/3 | trein

,resp_text,clarity
0,"Prezada Hellen, Dentre as pesquisas domici...",c234
1,"Senhor Francisco, O Serviço de Informações a...",c234
2,"Prezada Srª. Raylane, Em resposta a sua solic...",c5
3,"Prezada Senhora, De acordo com o art.13 do D...",c1
4,"Prezado(a) Senhor(a), Em atendimento ao ped...",c1


Linhas geradas: 900


## Arquivos gerados

Em `resultados_splade_v5/<assinatura>/`:

- `busca_interna_fold_XX.csv`: resultados internos por configuração;
- `fold_XX.json`: checkpoints dos folds externos concluídos;
- `folds_*.csv`: treino, validação, gap, C, normalização e convergência;
- `oof_*.csv`, `metricas_classes_*.csv` e `matriz_confusao_*.csv`;
- `comparacao_pareada.csv`: ganho por fold externo sobre o controle;
- `resumo_experimento.json`: configuração e métricas da avaliação;
- `busca_final.csv`, `configuracao_final.json` e `teste_rotulado_splade.xlsx`.

Em `cache_splade/` ficam as matrizes CSR do encoder, compatíveis com a V4.
